# 04 · Model training and selection

### Session 2 core notebook | Safiri Telecom churn case

---

We now have a clean analytical base table from notebook 02 and, from notebook 03, a
working idea of what complexity costs.

This notebook trains several model families properly and then makes the choice
between them. The selection step is where most of the value is, and it is not made
on AUC.

### Cell tags

| Tag | What you do |
|---|---|
| **READ** | Run it, read the output, answer the question. No typing. |
| **FILL** | Change one clearly marked value. |
| **BUILD** | Open-ended. Write your own code. |

## 0. Setup and the analytical base table

In [ ]:
import numpy as np
import pandas as pd

def make_panel(seed=42, n_subscribers=9000, n_months=8):
    """Simulate a monthly prepaid subscriber panel for Safiri Telecom."""
    rng = np.random.default_rng(seed)
    n = n_subscribers
    region  = rng.choice(["Nairobi","Coast","Western","Rift Valley","Eastern"], n, p=[.34,.16,.18,.20,.12])
    plan    = rng.choice(["Prepaid Basic","Prepaid Plus","Data Bundle"], n, p=[.55,.28,.17])
    handset = rng.choice(["Feature","Smart-Entry","Smart-Mid","Smart-High"], n, p=[.22,.38,.28,.12])
    tenure0, base_arpu = rng.integers(1,72,n), rng.lognormal(5.6,.55,n)
    heavy_data, eps = rng.random(n) < .30, rng.normal(0,.55,n)

    r1 = (-1.0 - 0.032*np.clip(tenure0,0,48) - 0.0014*base_arpu
          + 0.60*(plan=="Prepaid Basic") + 0.50*(handset=="Feature")
          + 0.30*(region=="Western") + eps)
    r2 = (-0.55 - 0.006*np.clip(tenure0,0,48) + 0.0003*base_arpu + 1.15*heavy_data
          + 0.70*np.isin(handset,["Smart-Mid","Smart-High"]) + 0.20*(plan=="Prepaid Basic") + eps)
    p1, p2 = 1/(1+np.exp(-r1)), 1/(1+np.exp(-r2))

    churn_month = np.full(n, 99)
    for m in range(1, n_months+1):
        hz = np.clip((p1 if m <= 5 else p2)*0.20, 0, .9)
        churn_month[(rng.random(n) < hz) & (churn_month == 99)] = m

    rows = []
    for m in range(1, n_months+1):
        alive = churn_month >= m
        decay = np.where(np.clip(churn_month-m,0,6) <= 2, 0.66 + 0.17*np.clip(churn_month-m,0,6), 1.0)
        rows.append(pd.DataFrame({
            "subscriber_id": np.arange(n), "month": m, "region": region, "plan": plan,
            "handset_class": handset, "tenure_months": tenure0 + m,
            "topup_kes": np.where(alive, base_arpu*decay*rng.lognormal(0,.30,n), 0).round(0),
            "topup_count": np.where(alive, rng.poisson(np.clip(6*decay,0.3,None)), 0),
            "data_mb": np.where(alive, base_arpu*decay*rng.lognormal(0,.5,n)*(1+1.4*heavy_data), 0).round(1),
            "voice_minutes": np.where(alive, rng.gamma(3, 22*decay), 0).round(1),
            "days_since_last_topup": np.where(alive,
                np.clip(rng.poisson(np.clip(20*(1-decay)+4,1,None)),0,60), 60),
            "complaints_90d": rng.poisson(0.18 + 0.45*((churn_month-m) <= 2), n),
            "active": alive.astype(int), "_churn_month": churn_month}))
    return pd.concat(rows, ignore_index=True)

def build_snapshot(panel, prediction_month, horizon=2):
    """Features as at the prediction point; label from the outcome window."""
    obs = panel[(panel.month == prediction_month) & (panel.active == 1)].copy()
    cm = obs._churn_month.values
    obs["churned"] = ((cm > prediction_month) & (cm <= prediction_month + horizon)).astype(int)
    return obs.drop(columns=["_churn_month", "active", "month"])

FEATURES_NUM = ["tenure_months","topup_kes","topup_count","data_mb",
                "voice_minutes","days_since_last_topup","complaints_90d"]
FEATURES_CAT = ["region","plan","handset_class"]

import matplotlib.pyplot as plt
import seaborn as sns
import os, pickle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (9, 4.5)

# Use the ABT from notebook 02 if it is here; otherwise rebuild it.
if os.path.exists("abt_train.csv") and os.path.exists("abt_test.csv"):
    train, test = pd.read_csv("abt_train.csv"), pd.read_csv("abt_test.csv")
    print("Loaded the analytical base table from notebook 02.")
else:
    panel = make_panel()
    train = pd.concat([build_snapshot(panel,3), build_snapshot(panel,4)], ignore_index=True)
    test  = build_snapshot(panel, 6)
    THRESH = {"heavy_data": train.data_mb.quantile(.75), "low_topup": train.topup_kes.quantile(.25)}
    for d in (train, test):
        d["heavy_data_user"]   = (d.data_mb   > THRESH["heavy_data"]).astype(int)
        d["low_value"]         = (d.topup_kes < THRESH["low_topup"]).astype(int)
        d["has_complained"]    = (d.complaints_90d >= 1).astype(int)
        d["lapsed"]            = (d.days_since_last_topup > 21).astype(int)
        d["revenue_per_topup"] = d.topup_kes / d.topup_count.clip(lower=1)
        d["smartphone"]        = d.handset_class.isin(["Smart-Mid","Smart-High"]).astype(int)
    print("Rebuilt the analytical base table.")

print(f"train {train.shape} churn {train.churned.mean():.1%}")
print(f"test  {test.shape} churn {test.churned.mean():.1%}")

## 1. The win condition, stated before training anything

In [ ]:
CALL_CAPACITY  = 500
OFFER_COST_KES = 200
MARGIN_KES     = 2400
TAKE_UP_RATE   = 0.35

breakeven = OFFER_COST_KES / (MARGIN_KES * TAKE_UP_RATE)

def precision_at_k(scores, truth, k=CALL_CAPACITY):
    idx = np.argsort(-np.asarray(scores))[:k]
    hits = np.asarray(truth)[idx]
    return hits.mean(), hits.sum()

def campaign_value(scores, truth, k=CALL_CAPACITY):
    _, hits = precision_at_k(scores, truth, k)
    return hits * TAKE_UP_RATE * MARGIN_KES - k * OFFER_COST_KES

print(f"Break-even precision: {breakeven:.1%}")
print(f"WIN CONDITION: beat the incumbent rule on net campaign value at k={CALL_CAPACITY}.")

The original version of this notebook asked *"does it satisfy our project's win
condition?"* without ever defining one. That question cannot be answered, and a
student who tries will fall back on whichever metric looks best.

Define it first, in currency, or the selection step is arbitrary.

## 2. The incumbent

In [ ]:
incumbent_scores = test.complaints_90d.values
p, hits = precision_at_k(incumbent_scores, test.churned.values)
INCUMBENT_VALUE = campaign_value(incumbent_scores, test.churned.values)

print(f"Incumbent rule: call whoever complained most")
print(f"  precision@{CALL_CAPACITY}: {p:.1%}   churners found: {hits}")
print(f"  net campaign value: KES {INCUMBENT_VALUE:,.0f}")
print(f"\nThis is the number to beat. Not 0.5 AUC.")

## 3. Splitting: temporal, and why it costs you

Our train and test already come from different months. The tempting alternative is
to pool them and split at random, which gives a better number and a wrong one.

In [ ]:
NUM = ["tenure_months","topup_kes","topup_count","data_mb","voice_minutes",
       "days_since_last_topup","complaints_90d","revenue_per_topup"]
FLAG = ["heavy_data_user","low_value","has_complained","lapsed","smartphone"]
CAT  = ["region","plan","handset_class"]
NUM  = [c for c in NUM if c in train.columns]
FLAG = [c for c in FLAG if c in train.columns]

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM + FLAG),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])

X_train, y_train = train[NUM + FLAG + CAT], train.churned
X_test,  y_test  = test[NUM + FLAG + CAT],  test.churned
print(f"{len(NUM+FLAG+CAT)} input features")

## 4. Pipelines

Every model goes inside a `Pipeline` with its preprocessing. This is not a style
preference. During cross-validation the scaler is refitted on each training fold
alone, so no fold ever sees the statistics of its own validation data.

Fit a scaler outside the pipeline and you have leaked, silently, into every fold.

In [ ]:
pipelines = {
    "logistic": Pipeline([("pre", make_preprocessor()),
                          ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))]),
    "rf":       Pipeline([("pre", make_preprocessor()),
                          ("clf", RandomForestClassifier(random_state=123, n_jobs=-1))]),
    "gb":       Pipeline([("pre", make_preprocessor()),
                          ("clf", GradientBoostingClassifier(random_state=123))]),
}

grids = {
    "logistic": {"clf__C": [0.05, 0.5, 5]},
    # 'auto' was REMOVED from max_features in scikit-learn 1.3. It raises
    # InvalidParameterError on any current install. Use 'sqrt', 'log2' or a float.
    "rf": {"clf__n_estimators": [200],
           "clf__max_features": ["sqrt", 0.33],
           "clf__min_samples_leaf": [5, 25]},
    "gb": {"clf__n_estimators": [150],
           "clf__learning_rate": [0.05, 0.1],
           "clf__max_depth": [2, 3]},
}
print("Total fits:", sum(np.prod([len(v) for v in g.values()]) for g in grids.values()) * 5)

**On grid size.** The original notebook ran a 5-model, 10-fold search over larger
grids: several hundred fits, which is minutes of a teaching session spent watching
a progress bar.

`cv=5` and small grids here. In production you widen them; in a classroom you do
not. Knowing which situation you are in is part of the skill.

## 5. Cross-validated search

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fitted = {}

for name, pipe in pipelines.items():
    search = GridSearchCV(pipe, grids[name], cv=cv, scoring="average_precision", n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = search
    print(f"{name:<10} best CV avg-precision {search.best_score_:.4f}   {search.best_params_}")

**Note the scoring metric.** `average_precision` (area under the precision-recall
curve), not accuracy and not ROC AUC.

With a 9% positive rate, accuracy is useless: predict "nobody churns" and you score
91%. ROC AUC is better but it averages over the whole ranking, including the 90% of
subscribers you will never contact. Average precision weights the top of the
ranking, which is the only part you act on.

## 6. Selection

In [ ]:
rows = []
for name, search in fitted.items():
    s = search.predict_proba(X_test)[:, 1]
    prec, hits = precision_at_k(s, y_test.values)
    rows.append({
        "model": name,
        "CV avg-prec": search.best_score_,
        "test ROC AUC": roc_auc_score(y_test, s),
        "test avg-prec": average_precision_score(y_test, s),
        f"precision@{CALL_CAPACITY}": prec,
        "net value KES": campaign_value(s, y_test.values),
    })
rows.append({"model": "INCUMBENT RULE", "CV avg-prec": np.nan,
             "test ROC AUC": roc_auc_score(y_test, incumbent_scores),
             "test avg-prec": average_precision_score(y_test, incumbent_scores),
             f"precision@{CALL_CAPACITY}": precision_at_k(incumbent_scores, y_test.values)[0],
             "net value KES": INCUMBENT_VALUE})

board = pd.DataFrame(rows).set_index("model").sort_values("net value KES", ascending=False)
print(board.round(4).to_string())

**READ the table column by column, then answer.**

Look at where the incumbent rule sits. It has the **worst ROC AUC of anything in the
table**, and it still beats a fully tuned logistic regression on net campaign value.

A metric that ranks a whiteboard rule below a trained model, when the rule makes more
money, is not measuring the thing you care about. That is the entire point of this
notebook.

### A note on notebook 00

In the lifecycle notebook the model lost to this same rule outright. Here it wins.
Nothing was fudged: the difference is the enrichment from notebook 02, a model family
suited to the problem, and cross-validated tuning.

That is the honest sequence. The first model lost to the baseline, you reported it,
you did more work, and the second one won. Both halves of that sentence matter.

> *Which model would you deploy, and would you deploy one at all?*

Two failure modes to avoid. Picking the top of the AUC column is the graduate's
answer. Picking whichever row happens to be top of the value column without checking
whether it beats the incumbent is the consultant's answer. Do neither.

## 7. The decision curve

In [ ]:
best_name = board.index[0] if board.index[0] != "INCUMBENT RULE" else board.index[1]
best_scores = fitted[best_name].predict_proba(X_test)[:, 1]

ks = np.arange(100, 4001, 100)
v_model = [campaign_value(best_scores, y_test.values, k) for k in ks]
v_rule  = [campaign_value(incumbent_scores, y_test.values, k) for k in ks]

plt.plot(ks, v_model, lw=2.5, color="#0F766E", label=f"{best_name}")
plt.plot(ks, v_rule,  lw=2.5, color="#B45309", label="incumbent rule")
plt.axhline(0, color="grey", ls=":")
plt.axvline(CALL_CAPACITY, color="#DC2626", ls="--", label=f"capacity = {CALL_CAPACITY}")
plt.xlabel("subscribers contacted"); plt.ylabel("net campaign value (KES)")
plt.title("The model is fixed. The operating point is a business decision.")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Best model: {best_name}")
print(f"  value at k={CALL_CAPACITY}: KES {campaign_value(best_scores, y_test.values):,.0f}")
print(f"  best value overall:  KES {max(v_model):,.0f} at k={ks[int(np.argmax(v_model))]}")
print(f"  incumbent rule best: KES {max(v_rule):,.0f} at k={ks[int(np.argmax(v_rule))]}")

## 8. Where does it fail?

In [ ]:
# Aggregate metrics hide segment failures. This is the cell that finds them.
seg = test.copy()
seg["score"] = best_scores
cut = np.sort(best_scores)[-CALL_CAPACITY]
seg["contacted"] = (seg.score >= cut).astype(int)

for col in ["region", "handset_class", "plan"]:
    g = seg.groupby(col).agg(subscribers=("churned","size"),
                             churn_rate=("churned","mean"),
                             contacted=("contacted","sum"))
    g["share_of_calls"] = g.contacted / g.contacted.sum()
    g["share_of_churn"] = (seg[seg.churned==1].groupby(col).size() / seg.churned.sum())
    print(g.round(3).to_string(), "\n")

**READ and answer.** Compare `share_of_calls` with `share_of_churn` for each segment.

> *Name one segment the model systematically under-serves. Is that a problem to fix,
> or a legitimate consequence of the cost model?*

Both answers can be right. What is never right is not knowing. A model that quietly
ignores an entire region is a finding you want to discover yourself, before a
regulator or a journalist does.

## 9. Save the model and its context

In [ ]:
artifact = {
    "model": fitted[best_name].best_estimator_,
    "model_family": best_name,
    "features": {"numeric": NUM, "flags": FLAG, "categorical": CAT},
    "trained_on": "prediction months 3-4",
    "evaluated_on": "prediction month 6",
    "label": "churned within 2 months of the prediction point",
    "operating_point": {"k": CALL_CAPACITY, "threshold": float(cut)},
    "economics": {"offer_cost": OFFER_COST_KES, "margin": MARGIN_KES,
                  "take_up": TAKE_UP_RATE, "breakeven_precision": breakeven},
    "incumbent_value_kes": float(INCUMBENT_VALUE),
}
with open("churn_model.pkl", "wb") as f:
    pickle.dump(artifact, f)
print("Saved churn_model.pkl")

**Never pickle a bare estimator.** The original notebook saved
`fitted_models['rf'].best_estimator_` and nothing else, which is the most common
handover failure in the industry: six months later nobody can say what it was
trained on, what the label meant, or where the threshold came from.

The estimator is the smallest part of what the next person needs.

**Next:** `05_Deployment_and_Monitoring`.